In [ ]:
# %%
# %%
# Finite-ensemble-size CRPS loss for the QRF-based stochastic ensemble product,
# reported in the units of the variable (degC) at a chosen set of ensemble
# sizes N (see ENSEMBLE_SIZES below).
#
# V3 CHANGE FROM V2: V2 solved for the N* that keeps the finite-ensemble
# penalty below a FRACTION of CRPS(inf):  N* = A / (tol * CRPS(inf)).
# That quantity depends only on the ratio A / CRPS(inf), which is ~1 for any
# roughly calibrated ensemble -- so N* ~ 1/tol whether the predictive
# distribution is wide (daily) or narrow (monthly), and it cannot show the
# effect of smoothing. The ABSOLUTE loss A/N (degC) does show it, because A
# itself shrinks when the predictive distribution narrows. V3 therefore
# reports, for each chosen N:
#     loss(N) = E[CRPS_hat(N)] - CRPS(inf) = A / N        (degC)
# and prints CRPS(inf) alongside.
#
# Method: the expected CRPS attained by a finite N-member ensemble degrades
# from its infinite-ensemble limit by a simple 1/N law:
#
#     E[CRPS_hat(N)] = CRPS(inf) + A/N
#
# This is the continuous-score analogue of two established discrete
# results: Richardson (2001, QJRMS 127:2473-2489) derived the equivalent
# E[Brier(m)] = Brier(inf) + A/m relation for the Brier score, and Tippett
# & Barnston (2008, generalizing Richardson 2001 and Mueller et al. 2005)
# showed the closed form E[RPS(m)] = (1 + 1/m) * E[RPS(inf)] for the ranked
# probability score. Ferro, Richardson & Weigel (2008, Meteorol. Appl.
# 15:19-24, doi:10.1002/met.45) extended this family of results to the
# CRPS and supplied a "fair" CRPS estimator that is UNBIASED for CRPS(inf)
# at any finite m. CRPS(inf) and A can therefore both be read directly off
# the EXISTING cross-validated ensemble in one pass.
#
# See also Leutbecher (2019, QJRMS 145:107-128, doi:10.1002/qj.3387).
#
# N larger than the number of members on disk (e.g. N=120, 150 when m=100;
# N=12, 15 when m=10) cannot be subsampled. Their loss is the A/N
# extrapolation, which is valid because the fair estimator of A is unbiased
# at any m -- provided additional members would behave like further iid draws
# from the same per-case predictive distribution. Those rows are flagged
# "extrapolated" in the printed table.
#
# Optional subsample check (RUN_SUBSAMPLE_CHECK): for N <= m, randomly draw N
# of the m members per case, compute the plain empirical CRPS, and report
# (mean plain CRPS - CRPS(inf)) next to A/N. In expectation this is an
# algebraic identity for subsets drawn without replacement, so it checks the
# implementation (fair-CRPS / A bookkeeping), NOT whether extra members would
# behave as iid draws.
#
# Worth stating plainly for the write-up: E-OBS's own headline ensemble
# size (100; Cornes et al. 2018, JGR-Atmos 123:9391-9409) was NOT derived
# this way -- their user guidance states the member count initially
# followed the precedent set by HadCRUT4's 100-member ensemble, to be
# refined later "by experimentation," with 100 also being a practically
# convenient number for reading off 5th/95th percentiles directly. Fine to
# cite as precedent, not a substitute for deriving N here.
#
# IMPORTANT ASSUMPTION (unchanged from V2) -- please sanity-check this once:
# the rasters being sampled here are the POST-Schaake-shuffle stochastic
# product, not the raw pre-shuffle QRF quantile draws. Schaake shuffle
# reorders which member INDEX is paired with which pixel/day to impose spatial
# structure -- it does not add, drop, or alter the SET of values realized at a
# given pixel-day. So the members sampled at one station on one day should be
# the same set (just arbitrarily labeled/ordered) as what the pre-shuffle QRF
# produced for that station-day, and are a valid input to fair CRPS. If your
# shuffle implementation does anything beyond a per-pixel relabeling (e.g.
# drops/resamples members), this assumption breaks and a separate pre-shuffle
# CV export would be needed instead.
#
# V3 - 2026-10-12 - jpk


# %%
import os
import json
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
TIMESCALE = 'daily'          # 'daily' or 'monthly' -- controls the raster
                              # filename pattern in stochastic_raster_path(),
                              # the CV period range, and which ENSEMBLE_SIZES
                              # list is reported

# Ensemble sizes at which to report the CRPS loss (degC)
ENSEMBLE_SIZES = {
    'daily':   [50, 80, 100, 120, 150],
    'monthly': [5, 8, 10, 12, 15],
}

RUN_SUBSAMPLE_CHECK = True   # empirical check of A/N for N <= m (see header)
N_SUBSAMPLE_REPS = 3         # random subsets averaged per N (each is already
                              # averaged over all CV cases, so 3 is plenty)
RANDOM_STATE = 64

START_YEAR, START_MONTH, START_DAY = 2001, 1, 1
END_YEAR, END_MONTH, END_DAY = 2025, 12, 31

# Same station / product paths as stochastic_ensemble_validation.py.
STN_PATHS = {
    "tmax": {
        "cleaned": r"F:\ClimateDataWS\Data\Insitu\Cleaned\tmax_cleaned.parquet",
        "test_stn": r"F:\ClimateDataWS\Data\Processed\Split_stations\tmax_test_stations.csv",
        "obs_col": "max_temp",
    },
    "tmin": {
        "cleaned": r"F:\ClimateDataWS\Data\Insitu\Cleaned\tmin_cleaned.parquet",
        "test_stn": r"F:\ClimateDataWS\Data\Processed\Split_stations\tmin_test_stations.csv",
        "obs_col": "min_temp",
    },
}

OUT_DIR = r"F:\ClimateDataWS\Data\Processed\Daily_bias_correction\EnsembleSize_diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)


def stochastic_raster_path(t_var, period):
    """Matches stochastic_ensemble_validation.py's daily naming pattern.
    TODO: confirm the monthly branch once the monthly stochastic product
    (1km QRF ensemble + MODIS LST predictors) actually exists on disk --
    this assumes one file per year-month, no day component, following the
    same {var}_stochastic_{...}.tif convention."""
    var_dir = t_var[:1].upper() + t_var[1:]
    if TIMESCALE == 'daily':
        fname = f"{t_var}_QRFSt_{period.year}{period.month:02d}{period.day:02d}.tif"
    elif TIMESCALE == 'monthly':
        fname = f"{t_var}_QRFSt_{period.year}{period.month:02d}.tif"
    else:
        raise ValueError(f"Unknown TIMESCALE: {TIMESCALE}")
    return fr"F:\ClimateDataWS\Data\Processed\Daily_bias_correction\QRF_Products\{var_dir}\Stochastic\{fname}"


# ---------------------------------------------------------------------------
# Fair CRPS (unbiased for CRPS(inf) at any finite m) and its 1/m penalty
# term -- Ferro, Richardson & Weigel (2008)
# ---------------------------------------------------------------------------
def fair_crps_and_penalty(ensemble, obs):
    """
    Returns (fair_crps, A) for one forecast case.
    fair_crps -- unbiased estimate of CRPS(inf); average over many cases to
                 get a stable CRPS(inf) estimate.
    A -- this case's contribution to the finite-ensemble penalty constant;
         average over many cases to get A, then E[CRPS_hat(N)] = CRPS(inf)
         + A/N follows directly, for ANY N (not only the m you have).
    """
    ensemble = np.asarray(ensemble, dtype=float)
    m = ensemble.size
    term1 = np.mean(np.abs(ensemble - obs))
    diffs = np.abs(ensemble[:, None] - ensemble[None, :])
    A = diffs.sum() / (2 * m * (m - 1))
    fair_crps = term1 - A
    return fair_crps, A


def plain_crps_vectorized(ens, obs):
    """
    Plain (biased-at-finite-N) empirical CRPS for many cases at once.
    ens: (n_cases, N) array, obs: (n_cases,) array -> (n_cases,) CRPS values.
    Pairwise term uses the sorted-sample identity
        sum_{i<j} |x_i - x_j| = sum_k (2k - N - 1) * x_(k)      (k = 1..N)
    which is O(N log N) per case instead of building the N x N matrix.
    CRPS_hat = mean|x - y| - (1 / (2 N^2)) * sum_{i,j} |x_i - x_j|
             = mean|x - y| - (sum_{i<j} |x_i - x_j|) / N^2
    """
    n = ens.shape[1]
    term1 = np.abs(ens - obs[:, None]).mean(axis=1)
    s = np.sort(ens, axis=1)
    w = 2 * np.arange(1, n + 1) - n - 1
    pair_sum = (s * w).sum(axis=1)
    return term1 - pair_sum / n ** 2


# ---------------------------------------------------------------------------
# Build the held-out-station comparison table directly from the stochastic
# rasters -- adapted from stochastic_ensemble_validation.py's Section A
# sampling loop, but keeping the FULL member array per case instead of only
# q5/q25/q50/q75/q95.
# ---------------------------------------------------------------------------
def build_comparison_df(t_var):
    cleaned = pd.read_parquet(STN_PATHS[t_var]["cleaned"])
    test_stn = pd.read_csv(STN_PATHS[t_var]["test_stn"])
    obs_col = STN_PATHS[t_var]["obs_col"]

    stn_points = list(zip(test_stn['long'], test_stn['lat']))

    if TIMESCALE == 'daily':
        periods = pd.period_range(
            start=f"{START_YEAR}-{START_MONTH:02d}-{START_DAY:02d}",
            end=f"{END_YEAR}-{END_MONTH:02d}-{END_DAY:02d}",
            freq="D",
        )
    else:  # monthly
        periods = pd.period_range(
            start=f"{START_YEAR}-{START_MONTH:02d}",
            end=f"{END_YEAR}-{END_MONTH:02d}",
            freq="M",
        )

    results = []
    missing_files = 0
    for period in tqdm(periods, desc=f"Sampling {TIMESCALE} stochastic rasters for {t_var}"):
        file_path = stochastic_raster_path(t_var, period)
        if not os.path.exists(file_path):
            missing_files += 1
            continue
        with rasterio.open(file_path) as src:
            if src.crs != "EPSG:4326":
                raise ValueError(
                    f"{file_path} is not EPSG:4326 -- station lon/lat sampling would be wrong"
                )
            sampled_values = list(src.sample(stn_points))
            day = getattr(period, "day", 1)
            for i, value in enumerate(sampled_values):
                arr = np.asarray(value, dtype=float)
                results.append({
                    'date': f'{period.year}-{period.month}-{day}',
                    'index_no': test_stn.iloc[i]['index_no'],
                    'ensemble': arr,
                    'n_members': arr.size,
                    'n_nodata': int(np.isnan(arr).sum()),
                })
    if missing_files:
        print(f"  [{t_var}] {missing_files} raster files not found on disk -- skipped")

    if not results:
        return pd.DataFrame()

    sampled_df = pd.DataFrame(results)
    sampled_df['date'] = pd.to_datetime(sampled_df['date'])

    test_ids = set(test_stn["index_no"])
    obs_df = cleaned[cleaned["index_no"].isin(test_ids)].copy()
    obs_df = obs_df.rename(columns={obs_col: 'observation'})

    comparison_df = sampled_df.merge(
        obs_df[['date', 'index_no', 'observation']], on=['date', 'index_no'], how='inner'
    )
    return comparison_df


def compute_fair_crps_table(comparison_df, expected_m):
    """Drops cases with any nodata pixel or an unexpected member count
    (e.g. a station falling just outside the raster extent on some days),
    then computes per-case fair CRPS and A on the rest."""
    n_before = len(comparison_df)
    clean = comparison_df[
        (comparison_df['n_nodata'] == 0) & (comparison_df['n_members'] == expected_m)
    ].copy()
    n_dropped = n_before - len(clean)
    if n_dropped:
        print(f"  dropped {n_dropped}/{n_before} cases with nodata or member-count mismatch "
              f"(expected m={expected_m})")

    fair_crps_vals = np.empty(len(clean))
    A_vals = np.empty(len(clean))
    for i, (_, row) in enumerate(clean.iterrows()):
        fair_crps_vals[i], A_vals[i] = fair_crps_and_penalty(row['ensemble'], row['observation'])
    clean = clean.assign(fair_crps=fair_crps_vals, A=A_vals)
    return clean


# ---------------------------------------------------------------------------
# Optional empirical check for N <= m (see header: this checks the
# implementation, it is not independent evidence about extra members)
# ---------------------------------------------------------------------------
def subsample_check(clean_df, sizes, m, CRPS_inf, n_reps=N_SUBSAMPLE_REPS, seed=RANDOM_STATE):
    """Returns {N: empirical loss in degC} for every N <= m in `sizes`.
    Loss = mean plain CRPS of random N-member subsets - CRPS(inf)."""
    ens = np.stack(clean_df['ensemble'].to_numpy())           # (n_cases, m)
    obs = clean_df['observation'].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)

    out = {}
    for N in sizes:
        if N > m:
            continue
        if N == m:
            crps_N = plain_crps_vectorized(ens, obs).mean()   # nothing to subsample
        else:
            crps_N = np.mean([
                plain_crps_vectorized(rng.permuted(ens, axis=1)[:, :N], obs).mean()
                for _ in range(n_reps)
            ])
        out[N] = float(crps_N - CRPS_inf)
    return out


def summarize_and_plot(t_var, clean_df, m):
    CRPS_inf = clean_df['fair_crps'].mean()
    A = clean_df['A'].mean()
    n_cases = len(clean_df)
    sizes = ENSEMBLE_SIZES[TIMESCALE]

    print(f"\n[{t_var}] {n_cases} cross-validation cases, {m} members/case")
    print(f"  CRPS(inf) estimate = {CRPS_inf:.4f} °C")
    print(f"  A (finite-ensemble penalty constant) = {A:.4f} °C")
    print(f"  Implied E[CRPS_hat(m={m})] = {CRPS_inf + A/m:.4f} °C  "
          f"(sanity check -- compare to your current plain empirical CRPS at m)")

    empirical = {}
    if RUN_SUBSAMPLE_CHECK:
        empirical = subsample_check(clean_df, sizes, m, CRPS_inf)

    print("  CRPS loss vs. infinite ensemble, loss(N) = A/N:")
    print(f"    {'N':>5} {'loss (°C)':>10} {'% of CRPS(inf)':>15} {'E[CRPS(N)] °C':>14} "
          f"{'subsample loss':>15}  note")
    for N in sizes:
        loss = A / N
        note = "current m" if N == m else ("extrapolated" if N > m else "")
        emp = f"{empirical[N]:15.4f}" if N in empirical else f"{'--':>15}"
        print(f"    {N:>5d} {loss:10.4f} {100 * loss / CRPS_inf:14.2f}% "
              f"{CRPS_inf + loss:14.4f} {emp}  {note}")

    results = {
        "variable": t_var, "timescale": TIMESCALE, "n_cases": int(n_cases),
        "m_current": int(m), "CRPS_inf_degC": float(CRPS_inf), "A_degC": float(A),
        "loss_by_N": {},
    }
    for N in sizes:
        loss = A / N
        results["loss_by_N"][str(N)] = {
            "loss_degC": float(loss),
            "loss_pct_of_CRPS_inf": float(100 * loss / CRPS_inf),
            "expected_crps_degC": float(CRPS_inf + loss),
            "extrapolated": bool(N > m),
            "subsample_loss_degC": empirical.get(N),
        }

    # ---- plot: E[CRPS_hat(N)] vs N, closed-form, with the chosen N marked ----
    N_lo = max(2, int(min(sizes) * 0.5))
    N_hi = int(max(sizes) * 1.25)
    N_range = np.arange(N_lo, N_hi + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(N_range, CRPS_inf + A / N_range, lw=1.5, label="CRPS(inf) + A/N")
    ax.axhline(CRPS_inf, color="grey", ls=":", lw=1, label=f"CRPS(inf) = {CRPS_inf:.3f} °C")
    ax.scatter(sizes, [CRPS_inf + A / N for N in sizes], color="C0", zorder=3,
               label="chosen N (annotated: loss, °C)")
    for N in sizes:
        ax.annotate(f"+{A / N:.4f}", (N, CRPS_inf + A / N), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=7)
    if empirical:
        ax.scatter(list(empirical), [CRPS_inf + v for v in empirical.values()],
                   facecolors="none", edgecolors="k", s=60, zorder=4, label="subsample check")
    ax.axvline(m, color="firebrick", lw=1.5, label=f"current m={m}")
    ax.set_xlabel("Ensemble size (N)")
    ax.set_ylabel("E[CRPS_hat(N)] (°C)")
    ax.set_title(f"{t_var} ({TIMESCALE}): expected CRPS vs. ensemble size")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"{t_var}_{TIMESCALE}_ensemble_size_crps_loss.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"  saved -> {plot_path}")

    summary_path = os.path.join(OUT_DIR, f"{t_var}_{TIMESCALE}_ensemble_size_loss_summary.json")
    with open(summary_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  summary -> {summary_path}")
    return results


# ---------------------------------------------------------------------------
# Main -- runs both tmax and tmin in one pass, matching
# stochastic_ensemble_validation.py's variable loop. Switch TIMESCALE to
# 'monthly' (and point the paths at the monthly product) for the monthly run.
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    if TIMESCALE not in ENSEMBLE_SIZES:
        raise ValueError(f"No ENSEMBLE_SIZES entry for TIMESCALE={TIMESCALE!r}")

    all_results = {}
    print(f"CRPS LOSS BY ENSEMBLE SIZE ({TIMESCALE}) ::::::::::::::")
    for t_var in ["tmax", "tmin"]:
#    for t_var in ["tmax"]:
        comparison_df = build_comparison_df(t_var)
        if comparison_df.empty:
            print(f"[{t_var}] no cases found -- check raster paths / date range / TIMESCALE")
            continue
        expected_m = int(comparison_df['n_members'].mode().iloc[0])
        clean_df = compute_fair_crps_table(comparison_df, expected_m=expected_m)
        all_results[t_var] = summarize_and_plot(t_var, clean_df, expected_m)